# 067 — Reconocimiento automático del habla

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Señal → features:** audio muestreado a 16 kHz → tramas de 25 ms (hop 10 ms) → FFT →
filtros en escala **mel** + log → log-mel espectrograma (entrada de Whisper, 80 bandas).
Los **MFCC** (DCT sobre el log-mel) fueron el estándar de la era HMM.

**Alineación:** ~300 tramas para ~10 palabras. **CTC** entrena sin alineación explícita:
emite símbolo o blanco `∅` por trama y suma la probabilidad de todas las alineaciones que
colapsan a la transcripción. **Whisper** (2022) usa en cambio un transformer
encoder-decoder que genera texto token a token, entrenado con 680 000 h de supervisión
débil: robusto sin fine-tuning, pero autoregresivo (lento) y capaz de alucinar texto en
silencios.

**Métrica:** `WER = (S+D+I)/N` con alineación de Levenshtein por palabra; puede superar
100 % y debe desglosarse por subgrupo de hablantes (acentos, edad, género).


### 🧮 Cálculo de referencia (para los ejercicios)

```text
ref (N=6): el modelo  transcribe la voz humana
hyp:       el modelos transcribe —  voz humana ya
S=1, D=1, I=1 → WER = 3/6 = 50 %

Tramas: 2 s a 16 kHz, ventana 25 ms, hop 10 ms → ≈198 tramas × 80 mel
```


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("perception", seed=67)
show(result)


## Reflexión

1. Tu ASR reporta WER 8 % global, pero el subtitulado falla sistemáticamente con hablantes
   andinos. ¿Qué desglose de evaluación faltó y qué datos corregirían el problema?
2. ¿Por qué la arquitectura autoregresiva de Whisper lo hace propenso a "transcribir"
   música o silencio, y qué componente previo del pipeline lo mitiga?
3. En una consulta médica transcrita, un WER de 5 % ¿es aceptable? ¿Qué palabras te
   preocupan más que el promedio y cómo las vigilarías?
